# [LAB 04] 13. 데이터 표준화

## #01. 준비작업

### 1. 라이브러리 참조

In [1]:
from jussam import load_data

# 스케일링을 위해 사용
from sklearn.preprocessing import MinMaxScaler, MaxAbsScaler, StandardScaler, RobustScaler

📦 연세대학교 주영아 교수가 제작한 라이브러리를 사용중입니다.
📧 Email(1): j.purplerose@yonsei.ac.kr
📧 Email(2): j.purplerose@gmail.com
📝 Website: https://juyounga.kr/


### 2. 샘플 데이터 가져오기

In [2]:
origin = load_data('nursing_grades')
origin

📚 어느 간호학과 대학원에 지원한 학생들에 대한 합격/불합격 여부를 조사한 가상의 데이터(메타데이터 없음)


,접수코드,이름,성별,필기점수,학부성적,병원경력,합격여부
0,NRS0001,장은우,남,380,3.610,3,불합격
1,NRS0002,최지호,남,660,3.670,3,합격
2,NRS0003,김하준,남,800,4.000,1,합격
3,NRS0004,임아윤,여,640,3.190,4,합격
4,NRS0005,강하준,남,520,2.930,4,불합격
...,...,...,...,...,...,...,...
395,NRS0396,박지유,여,620,4.000,2,불합격
396,NRS0397,조하은,여,560,3.040,3,불합격
397,NRS0398,박하윤,여,460,2.630,2,불합격
398,NRS0399,이지우,여,700,3.650,2,불합격


## #02. Min-Max Scaler

$정규화된 값 = (X - Xmin) / (Xmax - Xmin)$

- 모든 데이터의 범위를 `0~1`로 변환하는 것.
  - 데이터에서 최소값을 0으로, 최대값을 1로 매핑
  - 이 방법은 데이터의 분포를 유지하면서 데이터를 특정 범위로 축소시키는 데에 유용

### 1. 직접 계산하기

In [3]:
minmax_df = origin.copy()

x = minmax_df['필기점수']
xmin = x.min()
xmax = x.max()

minmax_df['필기점수_MinMax(1)'] = (x - xmin) / (xmax - xmin)
minmax_df.head()

,접수코드,이름,성별,필기점수,학부성적,병원경력,합격여부,필기점수_MinMax(1)
0,NRS0001,장은우,남,380,3.610,3,불합격,0.276
1,NRS0002,최지호,남,660,3.670,3,합격,0.759
2,NRS0003,김하준,남,800,4.000,1,합격,1.000
3,NRS0004,임아윤,여,640,3.190,4,합격,0.724
4,NRS0005,강하준,남,520,2.930,4,불합격,0.517


### 2. sklearn 활용

#### (1) 특정 필드만 추출

In [4]:
x = minmax_df[['필기점수']]
x.head()

,필기점수
0,380
1,660
2,800
3,640
4,520


#### (2) 스케일링 적용

In [5]:
scaler = MinMaxScaler()
minmax_df['필기점수_MinMax(2)'] = scaler.fit_transform(x)
minmax_df.head()

,접수코드,이름,성별,필기점수,학부성적,병원경력,합격여부,필기점수_MinMax(1),필기점수_MinMax(2)
0,NRS0001,장은우,남,380,3.610,3,불합격,0.276,0.276
1,NRS0002,최지호,남,660,3.670,3,합격,0.759,0.759
2,NRS0003,김하준,남,800,4.000,1,합격,1.000,1.000
3,NRS0004,임아윤,여,640,3.190,4,합격,0.724,0.724
4,NRS0005,강하준,남,520,2.930,4,불합격,0.517,0.517


## #03. Max-Abs Scaler

### 1. 직접 계산하기

$$x_{\text{scaled}} = \frac{x}{\max(|x|)}$$

- 절대값 기준으로 `-1~1` 범위로 스케일링
  - 음수/양수 모두 유지, 희소 행렬(sparse matrix)에 유리

In [6]:
abs_df = origin.copy()
x = abs_df['필기점수']
abs_df['필기점수_MaxAbs(1)'] = x / x.abs().max()
abs_df.head()

,접수코드,이름,성별,필기점수,학부성적,병원경력,합격여부,필기점수_MaxAbs(1)
0,NRS0001,장은우,남,380,3.610,3,불합격,0.475
1,NRS0002,최지호,남,660,3.670,3,합격,0.825
2,NRS0003,김하준,남,800,4.000,1,합격,1.000
3,NRS0004,임아윤,여,640,3.190,4,합격,0.800
4,NRS0005,강하준,남,520,2.930,4,불합격,0.650


### 2. Sklearn 활용

In [7]:
scaler = MaxAbsScaler()
x = abs_df[['필기점수']]
abs_df['필기점수_MaxAbs(2)'] = scaler.fit_transform(x)
abs_df.head()

,접수코드,이름,성별,필기점수,학부성적,병원경력,합격여부,필기점수_MaxAbs(1),필기점수_MaxAbs(2)
0,NRS0001,장은우,남,380,3.610,3,불합격,0.475,0.475
1,NRS0002,최지호,남,660,3.670,3,합격,0.825,0.825
2,NRS0003,김하준,남,800,4.000,1,합격,1.000,1.000
3,NRS0004,임아윤,여,640,3.190,4,합격,0.800,0.800
4,NRS0005,강하준,남,520,2.930,4,불합격,0.650,0.650


## #04. Standard Scaler (z-score, 표준화)

### 1. 직접 계산하기

$ 정규화된 값 = (X - 평균) / 표준편차 $

- 데이터를 평균이 `0`, 표준편차가 `1`인 표준정규분포를 따르도록 변환
  - 데이터를 정규분포에 근사시켜서 이상치에 덜 민감하게 만들어 줌

In [8]:
std_df = origin.copy()
x = origin['학부성적']
std_df['학부성적_Standard(1)'] = (x - x.mean()) / x.std()
std_df.head()

,접수코드,이름,성별,필기점수,학부성적,병원경력,합격여부,학부성적_Standard(1)
0,NRS0001,장은우,남,380,3.610,3,불합격,0.578
1,NRS0002,최지호,남,660,3.670,3,합격,0.736
2,NRS0003,김하준,남,800,4.000,1,합격,1.603
3,NRS0004,임아윤,여,640,3.190,4,합격,-0.525
4,NRS0005,강하준,남,520,2.930,4,불합격,-1.208


### 2. sklearn 활용

In [9]:
scaler = StandardScaler()
x = std_df[['학부성적']]
std_df['학부성적_Standard(2)'] = scaler.fit_transform(x)
std_df.head()

,접수코드,이름,성별,필기점수,학부성적,병원경력,합격여부,학부성적_Standard(1),학부성적_Standard(2)
0,NRS0001,장은우,남,380,3.610,3,불합격,0.578,0.579
1,NRS0002,최지호,남,660,3.670,3,합격,0.736,0.737
2,NRS0003,김하준,남,800,4.000,1,합격,1.603,1.605
3,NRS0004,임아윤,여,640,3.190,4,합격,-0.525,-0.526
4,NRS0005,강하준,남,520,2.930,4,불합격,-1.208,-1.210


## #05. Robust Scaler

### 1. 직접 계산하기

$ 정규화된 값 = (X - median) / iqr $

$ iqr = Q3 - Q1 $

- 이상치가 존재할 경우 사용하는 방법.
  - 이상치가 포함된 데이터를 표준화(Standardization)하거나 정규화(Normalization)할 때, 이상치의 영향으로 전체 데이터의 분포가 왜곡됨
  - RobustScaler는 이 문제를 해결하기 위해 중앙값과 사분위수를 사용하여 데이터를 스케일링 함

In [10]:
rb_df = origin.copy()

x = rb_df['병원경력']
m = x.median()
iqr = x.quantile(0.75) - x.quantile(0.25)

rb_df['병원경력_Robust(1)'] = (x - m) / iqr
rb_df.head()

,접수코드,이름,성별,필기점수,학부성적,병원경력,합격여부,병원경력_Robust(1)
0,NRS0001,장은우,남,380,3.610,3,불합격,1.000
1,NRS0002,최지호,남,660,3.670,3,합격,1.000
2,NRS0003,김하준,남,800,4.000,1,합격,-1.000
3,NRS0004,임아윤,여,640,3.190,4,합격,2.000
4,NRS0005,강하준,남,520,2.930,4,불합격,2.000


### 2. sklearn 활용

In [11]:
scaler = RobustScaler()
x = rb_df[['병원경력']]
rb_df['병원경력_Robust(2)'] = scaler.fit_transform(x)
rb_df.head()

,접수코드,이름,성별,필기점수,학부성적,병원경력,합격여부,병원경력_Robust(1),병원경력_Robust(2)
0,NRS0001,장은우,남,380,3.610,3,불합격,1.000,1.000
1,NRS0002,최지호,남,660,3.670,3,합격,1.000,1.000
2,NRS0003,김하준,남,800,4.000,1,합격,-1.000,-1.000
3,NRS0004,임아윤,여,640,3.190,4,합격,2.000,2.000
4,NRS0005,강하준,남,520,2.930,4,불합격,2.000,2.000
